Aims-vcr


In [4]:
!pip install -q git+https://github.com/huggingface/transformers accelerate qwen-vl-utils bitsandbytes peft trl

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [5]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [6]:
from google.colab import userdata
from huggingface_hub import login
import wandb

login(token=userdata.get("HF_TOKEN"))
wandb.login(key=userdata.get("WANDB_API_KEY"))

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tanygupt360 (tanygupt360-delhi-technological-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [7]:
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import PeftModel
from datasets import Dataset

model_name = "Qwen/Qwen3-VL-4B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
)
processor = AutoProcessor.from_pretrained(model_name)

# Loading my pretrained model for even greater yumminess
model = PeftModel.from_pretrained(
    model,
    "zzephyrr/qwen3vl-vcr-lora-v2",
    is_trainable=True,
)

model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.print_trainable_parameters()

config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/64.7k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

...

[ERROR] `min_frames` is part of Qwen3VLVideoProcessorInitKwargs, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.12/dist-packages/transformers/models/qwen3_vl/video_processing_qwen3_vl.py.
[ERROR] `max_frames` is part of Qwen3VLVideoProcessorInitKwargs, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.12/dist-packages/transformers/models/qwen3_vl/video_processing_qwen3_vl.py.


adapter_config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 33.1MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

trainable params: 16,515,072 || all params: 4,454,330,880 || trainable%: 0.3708


In [8]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3VLForConditionalGeneration(
      (model): Qwen3VLModel(
        (visual): Qwen3VLVisionModel(
          (patch_embed): Qwen3VLVisionPatchEmbed(
            (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
          )
          (pos_embed): Embedding(2304, 1024)
          (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
          (blocks): ModuleList(
            (0-23): 24 x Qwen3VLVisionBlock(
              (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
              (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
              (attn): Qwen3VLVisionAttention(
                (qkv): Linear4bit(in_features=1024, out_features=3072, bias=True)
                (proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
              )
              (mlp): Qwen3VLVisionMLP(
                (linear_fc1): Linear4bit(in_features=1024, out_features=4096, bias=True)
   

In [9]:
from PIL import Image, ImageDraw
import requests
from io import BytesIO
from trl import SFTConfig, SFTTrainer

In [10]:
correct_answers = 0
correct_rationales = 0
output = []
predicted_answer = ""
correct_answerRationales = 0

In [11]:
s_num = 1

In [12]:
from datasets import load_dataset

#Starting not from 0 because I'm thinking to use the ones before that as evaluation or validation data
train_question_db = load_dataset(
    "Rowan/vcr",
    name="questions",
    split="train[600:800]"
)

README.md:   0%|          | 0.00/1.80k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

questions/train-00000-of-00008.parquet: reconstructing file:   0%|          |  0.00B / 76.3MB            

questions/train-00000-of-00008.parquet: downloading bytes:           |  0.00B            

questions/train-00001-of-00008.parquet: reconstructing file:   0%|          |  0.00B / 76.8MB            

...

questions/test-00007-of-00008.parquet: reconstructing file:   0%|          |  0.00B / 8.97MB            

questions/test-00007-of-00008.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/212923 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/26534 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25263 [00:00<?, ? examples/s]

In [13]:
def aPrompt(sample):

  question = sample["question_text"]
  answers = sample["answer_choice_texts"]

  prompt = f"""
You are solving a Visual Commonsense Reasoning (VCR) multiple-choice question.

Your task is to identify the ONE answer choice that is best supported by the image and the question.

Follow this reasoning process internally:

1. Carefully inspect the image.
2. Identify the people, objects, actions, relationships, and relevant visual details.
3. Determine exactly what the question is asking.
4. Evaluate each answer choice against the visual evidence and the question.
5. Reject choices that contradict the image, answer a different question, or rely on unsupported assumptions.
6. Select the choice with the strongest evidence.

Important:
- Base your decision primarily on evidence visible in the image.
- Use ordinary commonsense only when it is necessary to interpret the situation.
- Do not choose an answer merely because it sounds plausible.
- Pay close attention to the distinction between similar people, objects, actions, and relationships.
- There is exactly ONE correct answer.

Question:
{question}

Answer choices:

A. {answers[0]}
B. {answers[1]}
C. {answers[2]}
D. {answers[3]}

After reasoning internally, respond with ONLY the letter of the correct choice:
A, B, C, or D.
"""
  return prompt
  # print(prompt)

In [14]:
def rPrompt(sample):
  global output
  global predicted_answer

  predicted_answer = output[0].strip()
  question = sample["question_text"]
  rationales = sample["rationale_choice_texts"]

  rationale_prompt = f"""
You are solving the rationale-selection stage of a Visual Commonsense Reasoning (VCR) task.

The model has already selected the following answer:

Selected answer:
{predicted_answer}

Your task is to identify the ONE rationale that best explains why that selected answer is correct.

Reason internally through the following steps:

1. Re-examine the image.
2. Consider the question and the selected answer together.
3. Determine what visual evidence or commonsense connection supports the selected answer.
4. Compare every rationale choice.
5. Reject rationales that:
   - contradict the image,
   - do not explain the selected answer,
   - refer to the wrong person/object/action,
   - introduce unsupported information,
   - or are logically weaker than another choice.
6. Select the rationale that provides the strongest explanation for the selected answer.

Important:
- The rationale must explain the SELECTED ANSWER, not merely describe the image.
- Do not choose a rationale just because it is generally plausible.
- Pay attention to people, actions, objects, and relationships mentioned in the question.
- There is exactly ONE correct rationale.

Question:
{question}

Selected answer:
{predicted_answer}

Rationale choices:

A. {rationales[0]}
B. {rationales[1]}
C. {rationales[2]}
D. {rationales[3]}

After reasoning internally, respond with ONLY the letter of the best rationale:
A, B, C, or D.
"""
  return rationale_prompt
  # print(rationale_prompt)

In [15]:
needed_images = set(train_question_db["img_fn"])
print(len(needed_images))

image_ds = load_dataset(
    "Rowan/vcr",
    name="image_examples",
    split="train",
    streaming = True
)

image_cache = {}
for img_sample in image_ds:
    fn = img_sample["img_fn"]
    if fn in needed_images:
        image_cache[fn] = img_sample
    if len(image_cache) == len(needed_images):
        break

79


Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

In [16]:
def preprocess_image(image_record):

    image = image_record["image"].convert("RGB")

    orig_w, orig_h = image.size

    image.thumbnail((768, 768))

    new_w, new_h = image.size

    scale_x = new_w / orig_w
    scale_y = new_h / orig_h

    boxes = []

    for box in image_record["boxes"]:
      x1, y1, x2, y2, confidence = box

      boxes.append([
          x1 * scale_x,
          y1 * scale_y,
          x2 * scale_x,
          y2 * scale_y,
          confidence
      ])

    draw = ImageDraw.Draw(image)

    for i, box in enumerate(boxes):

      x1, y1, x2, y2, confidence = box

      draw.rectangle(
          [x1, y1, x2, y2],
          outline="red",
          width=2
      )

      draw.text(
          ((x1+x2)/2, (y1+y2)/2),
          f"{image_record['objects'][i]}{i}",
          fill="green",
          font_size = 18
      )
    # display(image)
    return image

In [17]:
def build_example(sample, image_record):
  image = preprocess_image(image_record)

  correct_idx = sample["answer_label"]
  answer_letter = "ABCD"[correct_idx]

  return {
    "images": [image],
    "messages": [
      {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": aPrompt(sample)},
      ]},
      {"role": "assistant", "content": [
        {"type": "text", "text": answer_letter},
      ]},
    ]
  }

def build_rationale_example(sample, image_record):
  image = preprocess_image(image_record)

  correct_answer_idx = sample["answer_label"]
  correct_answer_text = sample["answer_choice_texts"][correct_answer_idx]
  correct_rationale_letter = "ABCD"[sample["rationale_label"]]

  # Build the rationale prompt using the KNOWN correct answer (not the model prediction)
  question = sample["question_text"]
  rationales = sample["rationale_choice_texts"]
  rationale_prompt = f"""
You are solving the rationale-selection stage of a Visual Commonsense Reasoning (VCR) task.

The model has already selected the following answer:

Selected answer:
{correct_answer_text}

Your task is to identify the ONE rationale that best explains why that selected answer is correct.

Rationale choices:

A. {rationales[0]}
B. {rationales[1]}
C. {rationales[2]}
D. {rationales[3]}

Respond with ONLY the letter of the correct rationale: A, B, C, or D.
"""

  return {
    "images": [image],
    "messages": [
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": rationale_prompt},
        ]},
        {"role": "assistant", "content": [
          {"type": "text", "text": correct_rationale_letter},
        ]},
      ]
  }

def build_joint_example(sample, image_record):
    image = preprocess_image(image_record)

    question = sample["question_text"]
    answers = sample["answer_choice_texts"]
    rationales = sample["rationale_choice_texts"]
    correct_answer_letter = "ABCD"[sample["answer_label"]]
    correct_rationale_letter = "ABCD"[sample["rationale_label"]]

    prompt = f"""
You are solving a Visual Commonsense Reasoning (VCR) task.

Question: {question}

Answer choices:
A. {answers[0]}
B. {answers[1]}
C. {answers[2]}
D. {answers[3]}

Rationale choices:
A. {rationales[0]}
B. {rationales[1]}
C. {rationales[2]}
D. {rationales[3]}

Select the correct answer AND the rationale that best justifies it.
Respond in exactly this format: "Answer: <letter>, Rationale: <letter>"
"""

    target = f"Answer: {correct_answer_letter}, Rationale: {correct_rationale_letter}"

    return {
        "images": [image],
        "messages": [
            {"role": "user", "content": [
                {"type": "image"},
                {"type": "text", "text": prompt},
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": target},
            ]},
        ]
    }

In [18]:
train_question_db

Dataset({
    features: ['movie', 'objects', 'interesting_scores', 'answer_likelihood', 'img_fn', 'metadata_fn', 'answer_orig', 'question_orig', 'rationale_orig', 'question_tokens', 'answer_choice_tokens', 'answer_label', 'answer_match_iter', 'answer_sources', 'rationale_choice_tokens', 'rationale_sources', 'rationale_match_iter', 'rationale_label', 'source_zip_crc_mismatch', 'img_id', 'question_number', 'annot_id', 'match_fold', 'match_index', 'question_text', 'answer_choice_texts', 'rationale_choice_texts'],
    num_rows: 200
})

In [19]:
formatted = []
for s in train_question_db:
    if s["img_fn"] not in image_cache:
        continue
    formatted.append(build_example(s, image_cache[s["img_fn"]]))            # answer task
    formatted.append(build_rationale_example(s, image_cache[s["img_fn"]]))  # rationale task
    formatted.append(build_joint_example(s, image_cache[s["img_fn"]]))

train_dataset = Dataset.from_list(formatted)

In [20]:
result = preprocess_image(image_cache[train_question_db[0]["img_fn"]])
print(type(result))

#it should print PIL.Image.Image, else error

<class 'PIL.Image.Image'>


In [21]:
model.train()
model.gradient_checkpointing_enable()
model.config.use_cache = False

from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
  output_dir="./qwen3vl-vcr-lora-v2",
  per_device_train_batch_size=1,
  gradient_accumulation_steps=1,
  max_steps=300,                      # number of stuff i need
  learning_rate=2e-4,
  bf16=True,
  fp16=False,
  optim="adamw_torch",
  gradient_checkpointing=True,
  logging_steps=5,
  save_strategy="steps",
  save_steps=20,                     # save at the end, yummerz
  remove_unused_columns=False,

  # Hugging Face Hub
  push_to_hub=True,
  hub_model_id="zzephyrr/qwen3vl-vcr-lora-v2",
  # W&B
  report_to="wandb",
  run_name="qwen3vl-4b-vcr-lora-t4-run2",
)

trainer = SFTTrainer(
  model=model,
  args=training_args,
  train_dataset=train_dataset,
  # train_dataset=rationale_dataset,
  processing_class=processor,
)
trainer.train()
trainer.push_to_hub()   # pushes the LoRA adapter (and model card) to the HFHub

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
5,4.475510
10,4.693323
15,4.582772
20,4.379106
25,4.082828
30,3.898135
35,4.359797
40,4.397978
45,4.601892
50,4.346433


No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/zzephyrr/qwen3vl-vcr-lora-v2/commit/bcc430ed7fb9776396b29f0ade62966eff45f0f1', commit_message='End of training', commit_description='', oid='bcc430ed7fb9776396b29f0ade62966eff45f0f1', pr_url=None, repo_url=RepoUrl('https://huggingface.co/zzephyrr/qwen3vl-vcr-lora-v2', endpoint='https://huggingface.co', repo_type='model', repo_id='zzephyrr/qwen3vl-vcr-lora-v2'), pr_revision=None, pr_num=None)

In [22]:
def output_generator(sample, image, prompt, answer_or_rationale):
  global output
  global predicted_answer
  global correct_answers
  global correct_rationales
  global correct_answerRationales
  global s_num
  answerTrue = False

  messages = [
      {"role": "user", "content": [
          {"type": "image", "image": image},
          {"type": "text", "text": prompt},
      ]}
  ]

  text = processor.apply_chat_template(
      messages, tokenize=False, add_generation_prompt=True
  )

  inputs = processor(
      text=[text], images=[image], padding=True, return_tensors="pt",
  ).to(model.device)

  with torch.no_grad():
      generated_ids = model.generate(
          **inputs,
          max_new_tokens=1,
          do_sample=False,
      )

  input_length = inputs["input_ids"].shape[1]
  generated_ids = generated_ids[:, input_length:]

  output = processor.batch_decode(generated_ids, skip_special_tokens=True)

  if answer_or_rationale:
    correct = "ABCD"[sample["answer_label"]]
    is_correct = output[0].strip() == correct
    print(f"[{s_num}] Answer  -> Model: {output[0].strip()!r}  Correct: {correct}  {'✅' if is_correct else '❌'}")
    s_num += 1
    if is_correct:
      answerTrue = True
      correct_answers += 1
  else:
    correct = "ABCD"[sample["rationale_label"]]
    is_correct = output[0].strip() == correct
    print(f"     Rationale -> Model: {output[0].strip()!r}  Correct: {correct}  {'✅' if is_correct else '❌'}")
    if is_correct:
      if answerTrue:
        correct_answerRationales += 1
      correct_rationales += 1

In [23]:
model.eval()
model.gradient_checkpointing_disable()
model.config.use_cache = True

In [24]:
from datasets import load_dataset

eval_question_db = load_dataset("Rowan/vcr", name="questions", split="train[:30]")  # notice i started my dataset from 400 while training, so the model doesnt know this exists :D

eval_needed_images = set(eval_question_db["img_fn"])
eval_image_ds = load_dataset("Rowan/vcr", name="image_examples", split="train", streaming=True)

eval_image_cache = {}
for img_sample in eval_image_ds:
    if img_sample["img_fn"] in eval_needed_images:
        eval_image_cache[img_sample["img_fn"]] = img_sample
    if len(eval_image_cache) == len(eval_needed_images):
        break

samples = list(eval_question_db)
image_cache = eval_image_cache   # output_generator/loop below reads from this name

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

In [25]:
correct_answers = 0
correct_rationales = 0
s_num = 1

for sample in samples:
  image_record = image_cache[sample["img_fn"]]
  image = preprocess_image(image_record)

  prompt_a = aPrompt(sample)
  output_generator(sample, image, prompt_a, True)
  prompt_r = rPrompt(sample)
  output_generator(sample, image, prompt_r, False)

n = len(samples)
print(f"\nAnswer accuracy:    {correct_answers}/{n} ({100*correct_answers/n:.1f}%)")
print(f"Rationale accuracy: {correct_rationales}/{n} ({100*correct_rationales/n:.1f}%)")
print(f"Answer and Rationale accuracy: {correct_answerRationales}/{n} ({100*correct_answerRationales/n:.1f}%)")

[1] Answer  -> Model: 'B'  Correct: B  ✅
     Rationale -> Model: 'B'  Correct: B  ✅
[2] Answer  -> Model: 'C'  Correct: C  ✅
     Rationale -> Model: 'C'  Correct: C  ✅
[3] Answer  -> Model: 'C'  Correct: C  ✅
     Rationale -> Model: 'B'  Correct: B  ✅
[4] Answer  -> Model: 'D'  Correct: D  ✅
     Rationale -> Model: 'A'  Correct: A  ✅
[5] Answer  -> Model: 'B'  Correct: C  ❌
     Rationale -> Model: 'C'  Correct: B  ❌
[6] Answer  -> Model: 'D'  Correct: D  ✅
     Rationale -> Model: 'C'  Correct: C  ✅
[7] Answer  -> Model: 'B'  Correct: B  ✅
     Rationale -> Model: 'B'  Correct: D  ❌
[8] Answer  -> Model: 'B'  Correct: B  ✅
     Rationale -> Model: 'B'  Correct: D  ❌
[9] Answer  -> Model: 'A'  Correct: C  ❌
     Rationale -> Model: 'B'  Correct: A  ❌
[10] Answer  -> Model: 'C'  Correct: C  ✅
     Rationale -> Model: 'B'  Correct: B  ✅
[11] Answer  -> Model: 'D'  Correct: D  ✅
     Rationale -> Model: 'D'  Correct: C  ❌
[12] Answer  -> Model: 'C'  Correct: A  ❌
     Rationale -> Mod

Answer and Rationale accuracy is 9/30 not 0/30, some error in the function prolly. Translates to roughly 30% Accuracy, let's go